# Read and Process Data

In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Customer_Data_Processing").getOrCreate()
spark

In [ ]:
df1=spark.read.format("csv").option("header","true").load("/data/customers_100.csv")

In [ ]:
df1.show(5)


In [ ]:
df1.printSchema()

In [ ]:
from pyspark.sql.functions import *


In [ ]:
df1=df1.withColumn("registration_date",to_date(col("registration_date"),"yyyy-MM-dd"))\
        .withColumn("is_active",col("is_active").cast("boolean"))

In [ ]:
df1.printSchema()

In [ ]:
df=df1.fillna({'city':'Unknown','state':'Unknown','country':'Unknown'})

In [ ]:
df=df.withColumn('registration_year',year(col('registration_date')))\
        .withColumn('registration_month',month(col('registration_date')))
df.show(5)

In [ ]:
unique_city=df.select(countDistinct('city')).collect()
unique_city[0][0]

In [ ]:
unique_state=df.select(countDistinct('state')).collect()
unique_state[0][0]

In [ ]:
unique_country=df.select(countDistinct('country')).collect()
unique_country[0][0]

In [ ]:
df.groupBy('city').count().orderBy(col('count').desc()).show()

In [ ]:
df.groupBy('city','country').count().orderBy(col('count').desc()).show()

# Pivot Table Count of Active & Inactive user per state


In [ ]:
df.groupBy('state').pivot('is_active').count().show()

In [ ]:
df.show(5)

# Finding Closest Registration Done

In [ ]:
df_recent_customers=df.filter(col('registration_date')>=lit('2023-07-01'))
df_recent_customers.show(10)
df_recent_customers.count()

# Olderst & Newest Customer per City

In [ ]:
df.groupBy('city').agg(min('registration_date').alias('Oldest'),max('registration_date').alias('Newest')).show()

In [ ]:
output_path="processed_customers"
df.write.mode('overwrite').parquet(output_path)

# Joining & Analyzing Customers & Orders

In [ ]:
orders_df=spark.read.format("csv").option("header","true").option('inferSchema','true').load("/data/orders.csv")

In [ ]:
orders_df.show(5)

In [ ]:
orders_df.printSchema()

# Analysis on Orders

In [ ]:
orders_df.select(
                round(avg('total_amount'),2).alias('Average'),
                round(max('total_amount'),2).alias('Maximum'),
                round(min('total_amount'),2).alias('Minimum')
                ).show()

In [ ]:
orders_df.groupBy('status').count().show()

In [ ]:
orders_df.groupBy('customer_id')\
    .agg(sum('total_amount').alias('Total_Spend_Per_Customer'))\
    .orderBy(col('Total_Spend_Per_Customer').desc())\
    .show(10)

# Applying joins

In [ ]:
customers_orders_df=df.join(orders_df,'customer_id','inner')

In [ ]:
customers_orders_df.count()

In [ ]:
customers_orders_df.show(10)

# Total Order Per Customer

In [ ]:
customer_order_count=customers_orders_df.groupBy('customer_id').count().orderBy(col('count').desc())
customer_order_count.show(3)

# Total Spend Per Customer

In [ ]:
spend_per_customer=customers_orders_df.groupBy('customer_id').agg(sum('total_amount').alias('Total_spend')).orderBy(col('Total_spend').desc())
spend_per_customer.show(3)

# Average spend per customer

In [ ]:
averagespend_per_customer=customers_orders_df.groupBy('customer_id').agg(avg('total_amount').alias('average_spend')).orderBy(col('average_spend').desc())
averagespend_per_customer.show(6)

# OrderBy Status

In [ ]:
status_df=customers_orders_df.groupBy('status').count()
status_df.show()

# Order By Month

In [ ]:
orderByMonth=customers_orders_df.withColumn('order_month',month(col('order_date')))\
        .groupBy('order_month')\
            .count()\
                .orderBy('order_month')
orderByMonth.show(4)

# Basic Window Operation

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank


window_spec = Window.orderBy(col('Total_spend').desc())

ranked_customers = spend_per_customer.withColumn('dense_rank', dense_rank().over(window_spec))
ranked_customers.show(10)


# Finding Customers  with High order Frequency But low Total spend

In [ ]:
customer_spend_vs_order=customer_order_count.join(spend_per_customer,'customer_id','inner')\
        .orderBy(col('count').desc(),col('Total_spend').desc())
customer_spend_vs_order.show(5)

In [ ]:
output_path='/data/final_customer_data'
customer_spend_vs_order.write.mode('overwrite').parquet(output_path)

In [ ]:
spark.stop()